<h1> Parsing + Chunking </h1>

In [1]:
from unstructured.partition.text import partition_text #https://docs.unstructured.io/open-source/core-functionality/partitioning #txt files
from unstructured.partition.pdf import partition_pdf #pdf files
from unstructured.partition.docx import partition_docx #word files

from unstructured.chunking.title import chunk_by_title #https://docs.unstructured.io/open-source/core-functionality/chunking
from unstructured.chunking.basic import chunk_elements

from pathlib import Path
from langchain_core.documents import Document

c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = Path (r"C:\Users\Admin\Desktop\ip\How To RAG\docs")

documentos = []
chunk_id = 1

for docs in path.iterdir():

    file_dtype = docs.suffix.lower()

    if file_dtype == ".pdf":

        parsing = partition_pdf (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)

    elif file_dtype == ".txt":

        parsing = partition_text (str(docs), languages = ["por"])
        chunks = chunk_elements (parsing, max_characters = 750)

    elif file_dtype == ".docx":        
        
        parsing = partition_docx (str(docs), languages = ["por"])
        chunks = chunk_by_title (parsing, max_characters = 750)


    for chunk in chunks:

        documentos.append (
            Document (
                page_content = chunk.text,
                metadata = {
                    "title": docs.name,
                    "chunk_id": chunk_id
                }
            )
        )

        chunk_id += 1


#documentos


<hr>

<h1> Retrieval</h1>

<h3> Sparse Retrieval </h3>

In [3]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 15) #@K aqui definido

<hr>

<h3> Dense Retrieval </h3>

<h5> Embeddings & Indexing </h5>

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção


emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1701.00it/s]


In [5]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

<h3> Reciprocal Rank Fusion </h3>

$$
RRF (d) = \sum _{i = 1}^{n} \frac {1} {k + pos_i(d)} 
$$

In [7]:
from eval.HitRate import Reciprocal_Rank_Fusion

### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query) #Procura Lexical
d_retrieval = dense_retrieval.invoke (query) #Procura Vetorial

#display (s_retrieval)
#display (d_retrieval)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

#for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
#    print (ordem)
#    print (doc_name)
#    print (chunk_id)
#    print (chunk_text)
#    print (score)

#display (len(teste))
#display (teste)


<h2> Eval Retrieval </h2>

In [6]:
#Dataset
import json

with open (r"C:\Users\Admin\Desktop\ip\How To RAG\eval\dataset.json", "r", encoding = "utf-8") as f:
    dataset = json.load(f)

from eval.HitRate import hitrate_k_sparse_retrieval, hitrate_k_dense_retrieval, hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_sparse_retrieval, mrr_dense_retrieval, mrr_hybrid_retrieval


<h3> HitRate@5 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 5)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 5
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@10 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> HitRate@15 </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 15)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

x = hitrate_k_sparse_retrieval (sparse_retrieval_1, dataset)
y = hitrate_k_dense_retrieval (dense_retrieval_1, dataset)
z = hitrate_k_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<h3> MRR </h3>

In [ ]:
#### Definição dos Retrievals ####
sparse_retrieval_1 = BM25Retriever.from_documents (documentos, k = 10)

dense_retrieval_1 = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

x = mrr_sparse_retrieval (sparse_retrieval_1, dataset)
y = mrr_dense_retrieval (dense_retrieval_1, dataset)
z = mrr_hybrid_retrieval (sparse_retrieval_1, dense_retrieval_1, dataset)

display (x)
display (y)
display (z)

<hr>

<h6> código solto </h6>

In [ ]:
from langchain_community.vectorstores import FAISS #https://docs.langchain.com/oss/python/integrations/vectorstores #Não o melhor para produção
from langchain_huggingface import HuggingFaceEmbeddings #https://reference.langchain.com/python/langchain-huggingface

In [ ]:
emb_hf = HuggingFaceEmbeddings (model_name = r"C:\Users\Admin\Desktop\models\Bi Encoder")

db = FAISS.from_documents (documentos, emb_hf)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 10
    }
)

In [ ]:
from langchain_community.retrievers import BM25Retriever #https://reference.langchain.com/python/langchain-community/retrievers

sparse_retrieval = BM25Retriever.from_documents (documentos, k = 10)

In [ ]:
dense_retrieval = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 20
    }
)

#x = dense_retrieval.invoke ("O que é a corrente dinâmica estipulada (Idyn)?")

In [ ]:
### EXEMPLO

query = "Em que consiste o relatório da rede EU Kids Online ?"

s_retrieval = sparse_retrieval.invoke (query)
d_retrieval = dense_retrieval.invoke (query)

teste = Reciprocal_Rank_Fusion ([s_retrieval, d_retrieval])

for ordem, (doc_name, chunk_id, chunk_text, score) in enumerate (teste, start = 1):
    print (ordem)
    print (doc_name)
    print (chunk_id)
    print (chunk_text)
    print (score)

#display (teste)


<h1> Reranking </h1>

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path, device_map = "cuda")
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 562.81it/s]


<h4> Cross Encoder </h4>

In [ ]:

query_reranker = [query] * len (teste)
chunks = [t[2] for t in teste]
docs_names = [t[0] for t in teste]
chunks_ids = [t[1] for t in teste]

pares = list(zip(query_reranker, chunks)) # [query, chunks]
#print (pares)

inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
#print (inputs)

cross_encoder_model.eval()
with torch.no_grad():
    logits = cross_encoder_model (**inputs).logits
    #print (logits)

#Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
rerank = sorted(zip(docs_names, chunks_ids, chunks, logits.tolist()), key = lambda x: x[3], reverse = True) #x[3] define que arg é usado pelo reverse

#display (rerank)


<h4> Ranking </h4>

In [ ]:
final = [(docs, chunks, chunks_content, scores) for docs, chunks, chunks_content, scores in rerank [:5]]

#display (final[1])

<h3> Eval RAG + Reranking </h3>

In [ ]:
from eval.MeanReciprocalRank import mrr_ranker_sparse_system, mrr_ranker_dense_system, mrr_ranker_hybrid_system

f = mrr_ranker_sparse_system (sparse_retrieval, dataset)
h = mrr_ranker_dense_system (dense_retrieval, dataset)
i = mrr_ranker_hybrid_system (sparse_retrieval, dense_retrieval, dataset)

display (f)
display (h)
display (i)


<h1> Eval Pré LLM </h1>

In [ ]:

#Cross Encoder Model para Rerank
path = r"C:\Users\Admin\Desktop\models\Cross Encoder"

cross_encoder_model = AutoModelForSequenceClassification.from_pretrained (path)
cross_encoder_model_tokenization = AutoTokenizer.from_pretrained (path)

#Definição de Sparse Retrieval
sparse_retrieval_pre_llm = BM25Retriever.from_documents (documentos, k = 15) #@K aqui definido

#Definição de Dense Retrieval
dense_retrieval_pre_llm = db.as_retriever (
    search_type = "similarity", #similarity, mmr, similarity_score_threshold
    search_kwargs = {
        "k": 15
    }
)

dados = []

for data in dataset:

    q = data["query"]

    #print (q)
    sp_retr = sparse_retrieval_pre_llm.invoke (q)
    den_retr = dense_retrieval_pre_llm.invoke (q)
    
    rrf = Reciprocal_Rank_Fusion ([sp_retr, den_retr])

    query_reranker = [q] * len (rrf)
    chunks = [t[2] for t in rrf]
    docs_names = [t[0] for t in rrf]
    chunks_ids = [t[1] for t in rrf]

    pares = list(zip(query_reranker, chunks)) # [query, chunks]
    #print (pares)

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True)
    #print (inputs)

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits
        #print (logits)

    #Organiza os logits com os chunks correspondentes #Muita Atenção! Para calcular HitRate e MRR com as funções criadas é necessário estar no mesmo formato do output do RRF.
    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True) #x[3] define que arg é usado pelo reverse

    rank = [(chunks, scores) for chunks, scores in rerank [:8]]

    dados.append (q)
    dados.append (rank)


In [ ]:
dados

<h4> Context Recall </h4>
<p> Dos chunks fornecidos a resposta correta consiste em algum ? </p>

In [9]:
results = [1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

context_recall_final = sum(results) / len(results)
 
print (sum(results) / len(results))

0.8771929824561403


<h4> Context Precision </h4>
<p> Do número de chunks finais, quais são relevantes para o modelo ? </p>

In [10]:
results = [1/5, 2/5, 2/5, 0/5, 2/5, 0/5, 2/5, 4/5, 5/5, 3/5, 2/5, 2/5, 2/5, 2/5, 1/5, 4/5, 2/5, 4/5, 1/5, 3/5, 1/5]

context_precision_final = sum(results) / len(results)

print (sum(results) / len(results)) 

0.42857142857142855


<h3> LLM </h3>

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

path = r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3-Q4"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained (path, device_map = device)
tokenizer2 = AutoTokenizer.from_pretrained (path)

device


Loading weights: 100%|██████████| 291/291 [00:07<00:00, 39.24it/s]


'cuda'

In [ ]:
print (torch.__version__)
print (torch.version.cuda)

In [ ]:
prompt = "Em que consiste o relatório da rede EU Kids Online ?"


chat_template = f"""
<|im_start|>
És um assistente especializado em responder exclusivamente com base no contexto fornecido.

Regras:
1. Utiliza apenas a informação do Contexto.
2. Não inventes informação.
3. Não uses conhecimento externo.
4. Se a resposta não existir no contexto, responde exatamente:
   "Informação não encontrada na base de dados."
5. Responde de forma breve e direta.
6. Formata a resposta de forma clara.

Contexto:
{feed_llm}
<|im_end|>

<|im_start>
{prompt}
<|im_end|>

<|im_start|>
"""

model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

input_length = model_inputs["input_ids"].shape[1]
response_tokens = generated_ids[0][input_length:]

print(tokenizer2.decode(response_tokens, skip_special_tokens = True))

<h1> Answer Relevance </h1>

In [ ]:
dataa = []

for q in dataset[150:175]:
    #print (q)

    query = q["query"]

    sparse_ret = sparse_retrieval.invoke (query)
    dense_ret = dense_retrieval.invoke (query)

    rrf = Reciprocal_Rank_Fusion ([sparse_ret, dense_ret])

    query_reranker = [query] * len (rrf)
    chunks = [t[2] for t in rrf]

    pares = list(zip(query_reranker, chunks))

    inputs = cross_encoder_model_tokenization (pares, return_tensors = "pt", padding = True, truncation = True).to(device)

    cross_encoder_model.eval()
    with torch.no_grad():
        logits = cross_encoder_model (**inputs).logits

    rerank = sorted(zip(chunks, logits.tolist()), key = lambda x: x[1], reverse = True)
    #display (rerank)

    feed_llm = [chunks for chunks, scores in rerank [:5]]
    #display (feed_llm)
    
    contexto = "\n\n".join(feed_llm)

    #print (feed_llm)

    prompt = query

    ##Analisar isto
    chat_template = f"""
    <s>[INST]
    És um assistente especializado em responder exclusivamente com base no Contexto fornecido.

    Regras:
    1. Utiliza apenas a informação do Contexto.
    2. Não inventes informação.
    3. Não uses conhecimento externo.
    4. Se a resposta não existir no contexto, responde exatamente: "Informação não encontrada na base de dados."
    5. Responde de forma breve e direta.
    6. Formata a resposta de forma clara.
    
    Contexto:
    {contexto}
    </s>[/INST]
    
    <s>
    Pergunta:
    {prompt}
    </s>

    <s>
    """

    #print (chat_template)

    
    model_inputs = tokenizer2 (chat_template, return_tensors = "pt").to(device)

    generated_ids = model.generate(**model_inputs, max_new_tokens = 512)

    input_length = model_inputs["input_ids"].shape[1]
    response_tokens = generated_ids[0][input_length:]

    output = tokenizer2.decode(response_tokens, skip_special_tokens = True)
    #print (output)

    dataa.append (query)
    dataa.append (feed_llm)
    dataa.append (output)
    


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[tra

In [13]:
dataa

['Quais ferramentas digitais usa Joana (17 anos)?',
 ['“Troca muitas vezes. Algumas vezes você pede uma coisa e parece que ele ignora completamente. Tipo, eu quero que você faça, sei lá, num tipo de texto não tão formal. E ele não consegue. Ele sempre deixa muito formalizado. Ele também tem um problema muito grande com pontuação. Ele sempre usa muitos pontos finais. Quase não usa vírgulas. Isso é uma coisa que eu acho que eles deveriam melhorar.” (Joana, 17)\n\nProcedimentos ‘automatizados’ na realização de trabalhos de casa mobilizam várias ferramentas. Nesta descrição sequencial, a rapidez, eficiência e estética da apresentação parecem superar o investimento cognitivo:',
  'Lançado no mês da Internet Segura, este relatório temático nacional visa mapear acessos, usos e experiências de crianças e jovens com IA generativa e perceber se e como esta se integra atualmente nas suas vidas digitais. Inclui dados estatísticos do inquérito nacional (2.111 crianças e jovens de 9 a 17 anos) e do 

In [14]:
import pandas as pd

df = pd.DataFrame (dataa, columns = ["x"])

df.to_csv ("Mistral-7B-Instruct-v0.3-Q4 RAG System Analysys (TESTE com System Prompt diferente).csv", index = False)
#dataa

<h3> Answer Relevance </h3>

<p> Avalia até que ponto a resposta responde à pergunta do utilizador. </p>
<p> Ajuda do GPT </p>
<p> Qwen2.5-0.5B-Instruct </p>

In [51]:
answer_rel = [0.65, 0.60, 0.40, 0.85, 0.90, 0.25, 0.55, 0.85, 0.90, 0.80, 0.70, 0.75, 0.85, 0.95, 0.55, 0.90, 0.55, 0.80, 0.90, 0.90, 0.85, 0.70, 0.60, 0.45, 0.55]

answer_rel_final = sum(answer_rel) / len(answer_rel)

print (len(answer_rel))
print (sum(answer_rel) / len(answer_rel))

25
0.7100000000000002


<h3> Faithfulness </h3>
<p> Avalia se a resposta está suportada pelos documentos recuperados pelo sistema RAG. </p>
<p> Explicar que os modelos podem inventar informação e isso tem que ter menos pontos. </p>
<p> Qwen2.5-0.5B-Instruct </p>

In [52]:
faith = [1, 1, 0, 0.70, 0.95, 0, 0, 0.15, 1, 0.30, 0.10, 0.20, 0.10, 0.60, 0.10, 0, 0, 0.10, 0.30, 0.85, 0, 0, 0, 0.20, 0.65]

faith_final = sum(faith) / len(faith) 

print (len(faith))
print (sum(faith) / len(faith))

25
0.33199999999999996


<hr>

<h1> RAGAS Score de um Sistema RAG </h1>

<p> Hybrid Retrieval -> Reranker -> Modelo (Qwen2.5-0.5B-Instruct) </p>

In [29]:
from eval.HitRate import hitrate_k_hybrid_retrieval
from eval.MeanReciprocalRank import mrr_hybrid_retrieval, mrr_ranker_hybrid_system


hitrate = hitrate_k_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)
mrr_hybrid = mrr_hybrid_retrieval (sparse_retrieval, dense_retrieval, dataset)
mrr_ranker_hybrid = mrr_ranker_hybrid_system (sparse_retrieval, dense_retrieval, dataset)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2114.79it/s]


In [ ]:
retr_score = mrr_ranker_hybrid["Mean Reciprocal Rank Chunks Rerank"]
context_score = (context_recall_final * 0.5) + (context_precision_final * 0.5)
gen_score = (answer_rel_final * 0.5) + (faith_final * 0.5)

#print (gen_score)

#display (hitrate)
#display (mrr_hybrid)
#display (mrr_ranker_hybrid)

RAGAS = (retr_score * 0.30) + (context_score * 0.30) + (gen_score * 0.40) * 100

print (f"{RAGAS:.2f}")


0.5210000000000001
21.20


In [57]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained ("mistralai/Mistral-7B-Instruct-v0.3")
tokenization = AutoTokenizer.from_pretrained ("mistralai/Mistral-7B-Instruct-v0.3")

model.save_pretrained (r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3")
tokenization.save_pretrained (r"C:\Users\Admin\Desktop\models\Mistral-7B-Instruct-v0.3")


c:\Users\Admin\Desktop\ip\How To RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Writing model shards: 100%|██████████| 1/1 [00:27<00:00, 27.59s/it]


('C:\\Users\\Admin\\Desktop\\models\\Mistral-7B-Instruct-v0.3\\tokenizer_config.json',
 'C:\\Users\\Admin\\Desktop\\models\\Mistral-7B-Instruct-v0.3\\chat_template.jinja',
 'C:\\Users\\Admin\\Desktop\\models\\Mistral-7B-Instruct-v0.3\\tokenizer.json')

<h1> Answer Relevance </h1>
<p> Phi-3.5-mini-instruct-Q4 </p>

In [54]:
answer_rel = [0.72, 0.61, 0.58, 0.90, 0.35, 0.70, 0.30, 0.88, 1, 0.85, 0.20, 0.86, 0.88, 0.91, 0.45, 0.93, 0.35, 0.80, 0.58, 0.95, 0.96, 0.92, 0.88, 0.55, 0.95]

answer_rel_final = sum(answer_rel) / len(answer_rel)

answer_rel_final

0.7223999999999999

<h1> Faithfulness </h1>

<p> Phi-3.5-mini-instruct-Q4 </p>

In [55]:
faith = [0.86, 0.78, 0.80, 0.97, 0.60, 0.90, 1, 0.72, 1, 0.95, 0.50, 0.55, 0.93, 0.78, 0.50, 0.96, 0.65, 0.88, 0.62, 0.98, 0.97, 0.98, 0.95, 0.85, 0.98]

faith_final = sum(faith) / len(faith)

faith_final

0.8264000000000001

<h1> RAGAS Score de um Sistema RAG </h1>

<p> Hybrid Retrieval -> Reranker -> Modelo (microsoft/Phi-3.5-mini-instruct) </p>

In [ ]:
gen_score_2 = (answer_rel_final * 0.5) + (faith_final * 0.5)

#print (gen_score_2)

RAGAS_2 = (retr_score * 0.30) + (context_score * 0.30) + (gen_score_2 * 0.40) * 100

display (RAGAS_2)

0.7744


31.33438419290414